In [ ]:

import os
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

#  import of psnr and ssim
try:
    from skimage.metrics import peak_signal_noise_ratio as sk_psnr, structural_similarity as sk_ssim
except Exception:
    raise RuntimeError("Install scikit-image: pip install scikit-image")

# pydicom to read .ima files
try:
    import pydicom
    HAS_PYDICOM = True
except Exception:
    HAS_PYDICOM = False
    print("pydicom not found — .IMA/.dcm will be attempted with pydicom disabled. Install with pip/conda for full support.")


# User configuration path of dataset

TRAIN_CLEAN_PATH = r"C:\Users\ankit\Downloads\3mm Slice Thickness\Sharp Kernel (D45)\L506"
INFERENCE_NOISY_PATH = r"C:\Users\ankit\Downloads\3mm Slice Thickness\Soft Kernel (B30)\L506"
OUTPUT_PATH = "./enhanced_results"
os.makedirs(OUTPUT_PATH, exist_ok=True)

BATCH_SIZE = 8
IMAGE_SIZE = 256
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BEST_MODEL_PATH = "best_redcnn_by_psnr.pth"

print("Device:", DEVICE)


# Utility: noise simulation

def simulate_low_dose_noise(image_tensor, photon_count=300.0, gauss_sigma=0.05):
    """
    image_tensor: torch tensor 1xHxW in [0,1]
    photon_count: lower -> stronger Poisson noise
    gauss_sigma: additional Gaussian noise on normalized scale
    """
    x = image_tensor.float().clamp(0.0, 1.0)
    counts = x * photon_count
    noisy_counts = torch.poisson(counts)
    noisy = noisy_counts / (photon_count + 1e-9)
    noisy = noisy + torch.randn_like(noisy) * gauss_sigma
    noisy = noisy.clamp(0.0, 1.0)
    return noisy


# Robust reader: pydicom first, then PIL

def read_ima_or_image(path):
    p = Path(path)
    # try pydicom
    if HAS_PYDICOM:
        try:
            ds = pydicom.dcmread(str(p), force=True)
            if hasattr(ds, "PixelData"):
                arr = ds.pixel_array.astype(np.float32)
                arr = np.clip(arr, -1000, 2000)  # HU clipping (medical CT)
                mn, mx = arr.min(), arr.max()
                if mx > mn:
                    arr = (arr - mn) / (mx - mn)
                else:
                    arr = np.zeros_like(arr)
                return arr
        except Exception:
            pass
    #  PIL
    try:
        with Image.open(str(p)) as im:
            im = im.convert("L")
            arr = np.array(im, dtype=np.float32)
            mn, mx = arr.min(), arr.max()
            if mx > mn:
                arr = (arr - mn) / (mx - mn)
            else:
                arr = np.zeros_like(arr)
            return arr
    except Exception:
        return None


# Dataset with augmentation

class CleanTrainingDataset(Dataset):
    def __init__(self, root_dir, img_size=256, augment=True):
        self.root_dir = Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Folder not found: {self.root_dir}")
        # collect recursively
        all_paths = [p for p in self.root_dir.rglob("*") if p.is_file()]
        self.files = sorted(all_paths)
        if len(self.files) == 0:
            raise RuntimeError(f"No files found under {self.root_dir}")
        self.img_size = img_size
        self.augment = augment
        self.tensor_resize = transforms.Resize((img_size, img_size))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        p = self.files[idx]
        arr = read_ima_or_image(p)
        if arr is None:
            # return zeros to avoid crashing;
            img = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
        else:
            t = torch.from_numpy(arr).unsqueeze(0).float()  # 1,H,W
            if t.shape[1] != self.img_size or t.shape[2] != self.img_size:
                t = self.tensor_resize(t)
            img = t
        # create noisy counterpart for training
        noisy = simulate_low_dose_noise(img, photon_count=300.0, gauss_sigma=0.05)
        
        if self.augment:
            if random.random() < 0.5:
                img = TF.hflip(img); noisy = TF.hflip(noisy)
            if random.random() < 0.5:
                img = TF.vflip(img); noisy = TF.vflip(noisy)
            rot = random.choice([0,1,2,3])
            if rot:
                img = torch.rot90(img, k=rot, dims=[1,2]); noisy = torch.rot90(noisy, k=rot, dims=[1,2])
        return noisy, img

class NoisyInferenceDataset(Dataset):
    def __init__(self, root_dir, img_size=256):
        self.root_dir = Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Folder not found: {self.root_dir}")
        all_paths = [p for p in self.root_dir.rglob("*") if p.is_file()]
        self.files = sorted(all_paths)
        if len(self.files) == 0:
            raise RuntimeError(f"No files found under {self.root_dir}")
        self.img_size = img_size
        self.tensor_resize = transforms.Resize((img_size, img_size))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        p = self.files[idx]
        arr = read_ima_or_image(p)
        if arr is None:
            tensor = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
        else:
            tensor = torch.from_numpy(arr).unsqueeze(0).float()
            if tensor.shape[1] != self.img_size or tensor.shape[2] != self.img_size:
                tensor = self.tensor_resize(tensor)
        return tensor, str(p)


# RED-CNN model 

class RED_CNN(nn.Module):
    def __init__(self):
        super(RED_CNN, self).__init__()
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(1, 48, 5, padding=2)
        self.conv2 = nn.Conv2d(48, 48, 5, padding=2)
        self.conv3 = nn.Conv2d(48, 48, 5, padding=2)
        self.tconv1 = nn.ConvTranspose2d(48, 48, 5, padding=2)
        self.tconv2 = nn.ConvTranspose2d(48, 48, 5, padding=2)
        self.final_conv = nn.Conv2d(48, 1, 5, padding=2)
    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))
        x4 = self.relu(self.tconv1(x3) + x2)
        x5 = self.relu(self.tconv2(x4) + x1)
        out = self.final_conv(x5)
        return out + x


#  psnr calculation

def window_for_display(arr):
    # arr in [0,1] -> 0..255 uint8
    a = np.clip(arr, 0.0, 1.0)
    return (a * 255).astype(np.uint8)

def show_compare(orig, enhanced, title_suffix=""):
    if torch.is_tensor(orig):
        orig = orig.squeeze().cpu().numpy()
    if torch.is_tensor(enhanced):
        enhanced = enhanced.squeeze().cpu().numpy()
    orig = np.clip(orig, 0.0, 1.0)
    enhanced = np.clip(enhanced, 0.0, 1.0)
    diff = np.abs(orig - enhanced)
    psnr = sk_psnr(orig, enhanced, data_range=1.0)
    try:
        ssim = sk_ssim(orig, enhanced, data_range=1.0)
    except Exception:
        ssim = float('nan')
    fig, ax = plt.subplots(1,4, figsize=(16,4))
    ax[0].imshow(window_for_display(orig), cmap='gray'); ax[0].set_title("Input")
    ax[1].imshow(window_for_display(enhanced), cmap='gray'); ax[1].set_title("Enhanced")
    ax[2].imshow(window_for_display(np.clip(diff*5,0,1)), cmap='hot'); ax[2].set_title("Diff x5")
    ax[3].imshow(window_for_display((orig+enhanced)/2), cmap='gray'); ax[3].set_title(f"PSNR={psnr:.2f}\nSSIM={ssim:.3f}")
    for a in ax: a.axis('off')
    plt.suptitle(title_suffix)
    plt.show()
    return psnr, ssim


# Training loop with PSNR

def train_model(model, train_path, val_split=0.1):
    # create dataset
    ds = CleanTrainingDataset(train_path, img_size=IMAGE_SIZE, augment=True)
    n_val = max(1, int(len(ds) * val_split))
    n_train = len(ds) - n_val
    train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    best_psnr = -1.0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running = 0.0
        for noisy, clean in train_loader:
            noisy = noisy.to(DEVICE); clean = clean.to(DEVICE)
            optimizer.zero_grad()
            out = model(noisy)
            loss = criterion(out, clean)
            loss.backward()
            optimizer.step()
            running += loss.item() * noisy.size(0)
        train_loss = running / len(train_loader.dataset)

        # validation: compute average PSNR on val set
        model.eval()
        psnr_vals = []
        ssim_vals = []
        with torch.no_grad():
            for noisy, clean in val_loader:
                noisy = noisy.to(DEVICE); clean = clean.to(DEVICE)
                out = model(noisy)
                out_np = out.squeeze().cpu().numpy()
                clean_np = clean.squeeze().cpu().numpy()
                psnr_vals.append(sk_psnr(clean_np, out_np, data_range=1.0))
                try:
                    ssim_vals.append(sk_ssim(clean_np, out_np, data_range=1.0))
                except Exception:
                    ssim_vals.append(np.nan)
        mean_psnr = float(np.nanmean(psnr_vals))
        mean_ssim = float(np.nanmean(ssim_vals))

        print(f"Epoch {epoch}/{NUM_EPOCHS} — TrainLoss: {train_loss:.6e}  ValPSNR: {mean_psnr:.3f}  ValSSIM: {mean_ssim:.4f}")

        # checkpoint by PSNR
        if mean_psnr > best_psnr:
            best_psnr = mean_psnr
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  -> New best PSNR {best_psnr:.3f}; saved {BEST_MODEL_PATH}")

    return model


# Inference

def run_inference_and_visualize(model, inference_path, save_outdir=OUTPUT_PATH, n_display=4, augment_strong=False):
    ds = NoisyInferenceDataset(inference_path, img_size=IMAGE_SIZE)
    loader = DataLoader(ds, batch_size=1, shuffle=False)
    # load best weights if exist
    if os.path.exists(BEST_MODEL_PATH):
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
        print("Loaded best model:", BEST_MODEL_PATH)
    model.to(DEVICE).eval()
    with torch.no_grad():
        for i, (noisy, path) in enumerate(loader):
            noisy = noisy.to(DEVICE)
            if augment_strong:
                noisy_vis = simulate_low_dose_noise(noisy, photon_count=150.0, gauss_sigma=0.08)
            else:
                noisy_vis = noisy
            out = model(noisy_vis)
            out_np = out.squeeze().cpu().numpy()
            noisy_np = noisy_vis.squeeze().cpu().numpy()
            # visualize
            print(f"[{i+1}] {path[0]}")
            psnr, ssim = show_compare(noisy_np, out_np, title_suffix=os.path.basename(path[0]))
            # save enhanced image as png
            save_name = os.path.splitext(os.path.basename(path[0]))[0]
            save_path = os.path.join(save_outdir, f"enhanced_{save_name}.png")
            plt.imsave(save_path, np.clip(out_np,0,1), cmap='gray')
            print("Saved:", save_path)
            if i >= n_display-1:
                break


# Run training + execute

if __name__ == "__main__":
    model = RED_CNN().to(DEVICE)
    print("Dataset sizes and sanity check:")
    try:
        ds_check = CleanTrainingDataset(TRAIN_CLEAN_PATH, img_size=IMAGE_SIZE, augment=False)
        print("  Found training files:", len(ds_check))
    except Exception as e:
        raise RuntimeError("Training dataset error: " + str(e))
    try:
        ds_inf = NoisyInferenceDataset(INFERENCE_NOISY_PATH, img_size=IMAGE_SIZE)
        print("  Found inference files:", len(ds_inf))
    except Exception as e:
        raise RuntimeError("Inference dataset error: " + str(e))

 
    model = train_model(model, TRAIN_CLEAN_PATH)

   
    run_inference_and_visualize(model, INFERENCE_NOISY_PATH, save_outdir=OUTPUT_PATH, n_display=4, augment_strong=False)

    print("Done. Enhanced images saved to:", OUTPUT_PATH)


Device: cpu
Dataset sizes and sanity check:
  Found training files: 211
  Found inference files: 211
Epoch 1/5 — TrainLoss: 3.446586e-03  ValPSNR: 25.581  ValSSIM: 0.4775
  -> New best PSNR 25.581; saved best_redcnn_by_psnr.pth
